In [2]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
df = pd.read_csv('ket_qua.csv')

Tổng hợp dữ liệu theo Ngày và SKU (Daily Roll-up)


In [4]:
# 3. Gom nhóm theo Ngày và Mã sản phẩm để tính tổng sản lượng
daily_sales = df.groupby(['Date', 'ItemCode'], as_index=False)['Quantity'].sum()
print(daily_sales.head(5))

         Date   ItemCode  Quantity
0  2020-11-17  SKU-08063        12
1  2020-11-17  SKU-09458       600
2  2020-11-18  SKU-08062         6
3  2020-11-18  SKU-09458       480
4  2020-11-20  SKU-09458       240


Thực hiện tạo lưới thời gian toàn vẹn (bao gồm cả các ngày không bán được sản phầm nào)

In [ ]:
# Tạo danh sách tất cả các ngày duy nhất từ mốc bắt đầu đến kết thúc
all_dates = pd.date_range(start=daily_sales['Date'].min(), end=daily_sales['Date'].max())
#Lấy danh sách tất cả các mã sản phẩm duy nhất
all_items = daily_sales['ItemCode'].unique()
# Tạo MultiIndex chứa mọi cặp tổ hợp (Date x SKU) có thể xảy ra
index = pd.MultiIndex.from_product([all_dates, all_items], names=['Date', 'ItemCode'])
grid_df = pd.DataFrame(index=index).reset_index()
print(grid_df.head(20))

         Date   ItemCode
0  2020-11-17  SKU-08063
1  2020-11-17  SKU-09458
2  2020-11-17  SKU-08062
3  2020-11-17  SKU-07675
4  2020-11-17  SKU-07678
5  2020-11-17  SKU-07679
6  2020-11-17  SKU-07688
7  2020-11-17  SKU-07689
8  2020-11-17  SKU-07690
9  2020-11-17  SKU-07691
10 2020-11-17  SKU-03349
11 2020-11-17  SKU-03351
12 2020-11-17  SKU-03687
13 2020-11-17  SKU-04396
14 2020-11-17  SKU-02764
15 2020-11-17  SKU-02774
16 2020-11-17  SKU-04394
17 2020-11-17  SKU-08191
18 2020-11-17  SKU-03526
19 2020-11-17  SKU-03914


Thực hiện ghép hai bảng Grif_df và  daily_sales

In [8]:
# Trộn lưới trống với dữ liệu bán hàng thực tế
daily_sales['Date'] = pd.to_datetime(daily_sales['Date'])
train_bi = pd.merge(grid_df, daily_sales, on=['Date', 'ItemCode'], how='left')
# Điền các ngày không có giao dịch bằng số 0
train_bi['Quantity'] = train_bi['Quantity'].fillna(0)

# Khống chế biên dưới: Lượng bán cuối cùng bắt buộc phải >= 0 theo quy định đề bài
train_bi['Quantity'] = train_bi['Quantity'].clip(lower=0)

# Tối ưu hóa dung lượng bộ nhớ để sẵn sàng cho bước tạo Feature tiếp theo
train_bi['Quantity'] = train_bi['Quantity'].astype(np.int32)
print(train_bi.head(10))

        Date   ItemCode  Quantity
0 2020-11-17  SKU-08063        12
1 2020-11-17  SKU-09458       600
2 2020-11-17  SKU-08062         0
3 2020-11-17  SKU-07675         0
4 2020-11-17  SKU-07678         0
5 2020-11-17  SKU-07679         0
6 2020-11-17  SKU-07688         0
7 2020-11-17  SKU-07689         0
8 2020-11-17  SKU-07690         0
9 2020-11-17  SKU-07691         0


In [13]:
#đọc bộ dữ liệu train_bi đên file csv
train_bi.to_csv('train_bi.csv', index=False)
